# EDA of MaNGA morphology metadata and compressed spectra

Colab-ready, memory-conscious EDA for:

- `morfologia.fits`, one metadata row per galaxy.
- A ZIP archive containing about 16,000 `.fits.gz` spectroscopic files.

The supplied spectrum examples contain `WAVE`, `FLUX_1RE`, `FLUX_FoV`, `ERROR_1RE`, and `ERROR_FoV`. The supplied metadata example contains `name`, `objra`, `objdec`, `Type`, `Ttype`, `nsa_z`, `LogMass`, `incl`, and `PETRO_TH90`.

The notebook reads one spectrum at a time from the ZIP and does not extract the full archive or retain all spectra in RAM.

In [ ]:
!pip -q install astropy pyarrow seaborn tqdm

In [ ]:
from google.colab import drive
from pathlib import Path
from io import BytesIO
import gzip, re, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from astropy.io import fits
from astropy.table import Table
from astropy.stats import sigma_clipped_stats

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore', category=RuntimeWarning)

## 1. Mount Google Drive and set paths

Edit the three paths. Outputs are written to a separate Drive folder. The ZIP is read in place.

In [ ]:
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
METADATA_FITS = DRIVE_ROOT / 'astronomy/morfologia.fits'
SPECTRA_ZIP = DRIVE_ROOT / 'astronomy/spectra.zip'
OUTPUT_DIR = DRIVE_ROOT / 'astronomy/eda_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for p in [METADATA_FITS, SPECTRA_ZIP, OUTPUT_DIR]:
    print(p, '->', 'exists' if p.exists() else 'NOT FOUND')

## 2. Simple astronomy I/O helpers

In [ ]:
def first_table_hdu(hdul):
    for hdu in hdul:
        if isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)) and hdu.data is not None:
            return hdu
    raise ValueError('No non-empty FITS table HDU found')


def table_to_dataframe(hdu):
    df = Table(hdu.data).to_pandas()
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].map(lambda x: x.decode('utf-8', errors='replace').strip()
                              if isinstance(x, (bytes, bytearray)) else x)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def find_column(df, wanted):
    clean = lambda x: re.sub(r'[^a-z0-9]', '', str(x).lower())
    lookup = {clean(c): c for c in df.columns}
    if clean(wanted) not in lookup:
        raise KeyError(f'{wanted!r} not found. Columns: {list(df.columns)}')
    return lookup[clean(wanted)]


def galaxy_key(value):
    name = Path(str(value)).name
    return re.sub(r'(?i)\.(fits\.gz|fits|csv)$', '', name).strip()


def read_spectrum_from_zip(zip_path, member):
    with zipfile.ZipFile(zip_path) as zf:
        raw = zf.read(member)
    if member.lower().endswith('.gz'):
        raw = gzip.decompress(raw)
    with fits.open(BytesIO(raw), memmap=False) as hdul:
        return table_to_dataframe(first_table_hdu(hdul))

## 3. Load and inspect morphology metadata

Common `-9999` sentinels are converted to missing values. The original `Type` and `Ttype` are preserved.

In [ ]:
with fits.open(METADATA_FITS, memmap=True) as hdul:
    hdul.info()
    metadata = table_to_dataframe(first_table_hdu(hdul))
metadata = metadata.replace([-9999, -9999.0, 9999, 9999.0], np.nan)
name_col = find_column(metadata, 'name')
type_col = find_column(metadata, 'Type')
ttype_col = find_column(metadata, 'Ttype')
metadata['galaxy_key'] = metadata[name_col].map(galaxy_key)
print('Shape:', metadata.shape)
print('Duplicate galaxy keys:', metadata['galaxy_key'].duplicated().sum())
display(metadata.head())
metadata.info()

## 4. Metadata EDA: missingness, classes, distributions, and correlations

In [ ]:
missing = metadata.isna().mean().mul(100).sort_values(ascending=False).rename('missing_percent')
display(missing.to_frame().head(25))
display(metadata[type_col].value_counts(dropna=False).to_frame('count').head(30))
num = metadata.select_dtypes(include=np.number)
display(num.describe().T.assign(missing=num.isna().sum()))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
metadata[type_col].value_counts().head(20).plot.bar(ax=axes[0], title='Top morphology labels')
sns.histplot(data=metadata, x=ttype_col, discrete=True, ax=axes[1])
axes[1].set_title('T-type distribution')
plt.tight_layout(); plt.show()

wanted = ['nsa_z', 'LogMass', 'incl', 'PETRO_TH90']
plot_cols = []
for x in wanted:
    try: plot_cols.append(find_column(metadata, x))
    except KeyError: pass
if plot_cols:
    metadata[plot_cols].hist(figsize=(14, 8), bins=40)
    plt.tight_layout(); plt.show()
    display(metadata[plot_cols].corr().round(3))

## 5. Early/late grouping

This notebook uses an explicit starting rule: **Early = `Ttype <= 0`**, **Late = `Ttype > 0`**, and missing `Ttype` = **Unknown**. Keep this threshold configurable, especially for S0/a and Sa transition objects.

In [ ]:
def broad_morphology(x):
    if pd.isna(x): return 'Unknown'
    return 'Early' if x <= 0 else 'Late'
metadata['broad_type'] = metadata[ttype_col].map(broad_morphology)
display(metadata['broad_type'].value_counts().to_frame('count'))
for wanted in ['LogMass', 'nsa_z', 'incl']:
    try:
        col = find_column(metadata, wanted)
        plt.figure(figsize=(7,4))
        sns.boxplot(data=metadata, x='broad_type', y=col, order=['Early','Late','Unknown'])
        plt.title(f'{col} by broad morphology'); plt.show()
    except KeyError: pass

## 6. Match ZIP members to metadata rows

In [ ]:
with zipfile.ZipFile(SPECTRA_ZIP) as zf:
    members = [x.filename for x in zf.infolist() if not x.is_dir()]
spectrum_members = [m for m in members if m.lower().endswith(('.fits.gz', '.fits'))]
member_index, duplicates = {}, []
for member in spectrum_members:
    key = galaxy_key(member)
    if key in member_index: duplicates.append((key, member_index[key], member))
    else: member_index[key] = member
metadata_keys = set(metadata['galaxy_key'].dropna())
spectrum_keys = set(member_index)
matched_keys = metadata_keys & spectrum_keys
report = pd.Series({
    'archive_members': len(members), 'spectrum_members': len(spectrum_members),
    'unique_spectrum_keys': len(spectrum_keys), 'metadata_keys': len(metadata_keys),
    'matched': len(matched_keys), 'metadata_without_spectrum': len(metadata_keys-spectrum_keys),
    'spectra_without_metadata': len(spectrum_keys-metadata_keys),
    'duplicate_spectrum_keys': len(duplicates)
})
display(report.to_frame('value'))

## 7. Inspect and plot one spectrum

In [ ]:
EXPECTED = ['WAVE','FLUX_1RE','FLUX_FoV','ERROR_1RE','ERROR_FoV']

def clean_spectrum(df):
    df = df.copy()
    cols = [find_column(df, c) for c in EXPECTED]
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
    wave = find_column(df, 'WAVE')
    return df.dropna(subset=[wave]).sort_values(wave)

example_key = sorted(matched_keys)[0]
example = clean_spectrum(read_spectrum_from_zip(SPECTRA_ZIP, member_index[example_key]))
print(example_key, example.shape, list(example.columns))
display(example.head())

wave, flux, err = [find_column(example, c) for c in ['WAVE','FLUX_1RE','ERROR_1RE']]
x, y, e = example[wave].to_numpy(), example[flux].to_numpy(), example[err].to_numpy()
good = np.isfinite(x) & np.isfinite(y)
band = good & np.isfinite(e) & (e >= 0)
plt.figure(figsize=(14,5)); plt.plot(x[good], y[good], lw=.8)
plt.fill_between(x[band], y[band]-e[band], y[band]+e[band], alpha=.18)
plt.title(example_key); plt.xlabel('Wavelength (native WAVE units)'); plt.ylabel('FLUX_1RE'); plt.show()

## 8. Sequential quality summaries

Run a pilot first. Then set `MAX_SPECTRA = None` for all matched files. Only compact statistics are accumulated.

In [ ]:
def summarize_spectrum(df, key):
    df = clean_spectrum(df); wcol = find_column(df,'WAVE'); w = df[wcol].to_numpy(float)
    out = {'galaxy_key':key, 'n_spectral_rows':len(df),
           'wave_min':np.nanmin(w) if len(w) else np.nan,
           'wave_max':np.nanmax(w) if len(w) else np.nan,
           'max_wave_gap':np.nanmax(np.diff(w)) if len(w)>1 else np.nan}
    for f_name, e_name, suffix in [('FLUX_1RE','ERROR_1RE','1RE'),('FLUX_FoV','ERROR_FoV','FoV')]:
        f = df[find_column(df,f_name)].to_numpy(float)
        e = df[find_column(df,e_name)].to_numpy(float)
        vf = np.isfinite(f); vs = vf & np.isfinite(e) & (e>0)
        _, med, std = sigma_clipped_stats(f[vf], sigma=3) if vf.any() else (np.nan,)*3
        out[f'missing_flux_{suffix}'] = 1-vf.mean() if len(f) else np.nan
        out[f'median_flux_{suffix}'] = med; out[f'robust_std_flux_{suffix}'] = std
        out[f'median_snr_{suffix}'] = np.nanmedian(f[vs]/e[vs]) if vs.any() else np.nan
    return out

MAX_SPECTRA = 200  # None for the full archive
keys = sorted(matched_keys) if MAX_SPECTRA is None else sorted(matched_keys)[:MAX_SPECTRA]
rows, failed = [], []
for key in tqdm(keys, desc='Spectra'):
    try: rows.append(summarize_spectrum(read_spectrum_from_zip(SPECTRA_ZIP, member_index[key]), key))
    except Exception as exc: failed.append({'galaxy_key':key, 'error':repr(exc)})
spectral_summary, failures = pd.DataFrame(rows), pd.DataFrame(failed)
eda = metadata.merge(spectral_summary, on='galaxy_key', how='left', validate='one_to_one')
print('Successful:',len(spectral_summary),'Failed:',len(failures)); display(spectral_summary.head())
spectral_summary.to_parquet(OUTPUT_DIR/'spectral_summary.parquet', index=False)
eda.to_parquet(OUTPUT_DIR/'morphology_spectral_eda.parquet', index=False)
failures.to_csv(OUTPUT_DIR/'spectrum_failures.csv', index=False)

## 9. Quality plots by Early/Late group

In [ ]:
metrics = [c for c in ['median_snr_1RE','median_snr_FoV','n_spectral_rows','max_wave_gap'] if c in eda]
if metrics:
    display(eda.groupby('broad_type')[metrics].agg(['count','median','mean','std']))
    fig, axes = plt.subplots(1,len(metrics),figsize=(6*len(metrics),4),squeeze=False)
    for ax, metric in zip(axes[0],metrics):
        sns.boxplot(data=eda,x='broad_type',y=metric,order=['Early','Late','Unknown'],ax=ax)
    plt.tight_layout(); plt.show()

## 10. Print a controlled number of spectra grouped by detailed `Type`

Printing every spectrum is not readable. Select actual labels from the metadata and a small number per label.

In [ ]:
def plot_examples_by_type(types_to_plot=None, n_per_type=2, flux_name='FLUX_1RE', seed=42):
    available = metadata[metadata['galaxy_key'].isin(matched_keys)]
    if types_to_plot is not None: available = available[available[type_col].isin(types_to_plot)]
    for morphology, group in available.groupby(type_col, dropna=False):
        sample = group.sample(min(n_per_type,len(group)), random_state=seed)
        fig, axes = plt.subplots(len(sample),1,figsize=(14,3.2*len(sample)),squeeze=False)
        for ax, (_, row) in zip(axes[:,0],sample.iterrows()):
            key=row['galaxy_key']; spec=clean_spectrum(read_spectrum_from_zip(SPECTRA_ZIP,member_index[key]))
            w=find_column(spec,'WAVE'); f=find_column(spec,flux_name)
            ax.plot(spec[w],spec[f],lw=.7); ax.set_title(f'{key} | Type={morphology} | Ttype={row[ttype_col]}')
            ax.set_xlabel('Wavelength'); ax.set_ylabel(flux_name)
        plt.tight_layout(); plt.show()

selected = [x for x in ['E','S0','Sa','Sb','Sc'] if x in set(metadata[type_col].dropna())]
plot_examples_by_type(selected, n_per_type=2)

## 11. Median normalized, rest-frame spectra by Early/Late group

Wavelengths are optionally shifted using `nsa_z`, interpolated to a common grid, and normalized by a configurable continuum window. Confirm units and choose a scientifically appropriate window from the FITS headers before interpreting the result.

In [ ]:
def normalized_on_grid(spec, z, grid, flux_name='FLUX_1RE', norm_window=(5450,5550)):
    spec=clean_spectrum(spec); w=spec[find_column(spec,'WAVE')].to_numpy(float); f=spec[find_column(spec,flux_name)].to_numpy(float)
    if np.isfinite(z) and z>-1: w=w/(1+z)
    good=np.isfinite(w)&np.isfinite(f); w,f=w[good],f[good]
    if len(w)<2: return None
    window=(w>=norm_window[0])&(w<=norm_window[1]); scale=np.nanmedian(f[window]) if window.any() else np.nanmedian(f)
    if not np.isfinite(scale) or scale==0: return None
    order=np.argsort(w); w,f=w[order],f[order]/scale; w,idx=np.unique(w,return_index=True)
    return np.interp(grid,w,f[idx],left=np.nan,right=np.nan)


def build_stack(group_name, grid, n=100, seed=42):
    zcol=find_column(metadata,'nsa_z')
    pool=metadata[(metadata.broad_type==group_name)&metadata.galaxy_key.isin(matched_keys)]
    pool=pool.sample(min(n,len(pool)),random_state=seed); stack=[]
    for _,row in tqdm(pool.iterrows(),total=len(pool),desc=group_name):
        try:
            y=normalized_on_grid(read_spectrum_from_zip(SPECTRA_ZIP,member_index[row.galaxy_key]),row[zcol],grid)
            if y is not None: stack.append(y)
        except Exception: pass
    return np.asarray(stack)

grid=np.arange(3700.,9000.,2.)
early_stack, late_stack = build_stack('Early',grid), build_stack('Late',grid)

def draw(ax,stack,label,color):
    if not stack.size: return
    med=np.nanmedian(stack,axis=0); lo=np.nanpercentile(stack,16,axis=0); hi=np.nanpercentile(stack,84,axis=0)
    ax.plot(grid,med,color=color,label=f'{label} median (n={len(stack)})'); ax.fill_between(grid,lo,hi,color=color,alpha=.18); ax.legend()
fig,axes=plt.subplots(2,1,figsize=(15,9),sharex=True)
draw(axes[0],early_stack,'Early','tab:red'); draw(axes[1],late_stack,'Late','tab:blue')
axes[0].set_title('Early type'); axes[1].set_title('Late type'); axes[1].set_xlabel('Rest-frame wavelength')
for ax in axes: ax.set_ylabel('Normalized FLUX_1RE')
plt.tight_layout(); plt.show()

## 12. How spectra and metadata relate to early/late classification

- `Type` is the detailed morphology label, while `Ttype` can support an ordered target or the explicit Early/Late rule used above.
- Metadata such as `LogMass`, `nsa_z`, inclination, angular size, and spectrum-quality metrics should be checked as potential confounders or selection effects.
- Spectral shape and line behavior can contain information associated with stellar populations, gas, and recent star formation. Differences cannot automatically be attributed only to morphology because mass, redshift, aperture, dust, inclination, and data quality can also contribute.
- Comparing `FLUX_1RE` with `FLUX_FoV` can reveal aperture-related differences, subject to the exact definitions in the source data.
- Keep Unknown and transition objects explicit rather than silently forcing them into a binary class.

### Recommended graphs grouped by `Type`

1. Two to five individual spectra per detailed Type, using identical axes.
2. Median normalized spectrum per Type with 16th–84th percentile bands and sample counts.
3. Early and Late median curves overlaid, plus a ratio or difference panel.
4. `FLUX_1RE` versus `FLUX_FoV`, or their ratio, by Type.
5. S/N, missing fraction, row count, and wavelength-gap boxes/violins by Type.
6. `LogMass`, `nsa_z`, inclination, and size by Type before interpreting spectral differences.
7. Early/Late spectra after matching or stratifying on mass and redshift.
8. Scientifically selected line-window zooms after confirming wavelength frame, units, masks, and continuum treatment.

### Before classification

Split by galaxy key before preprocessing; fit normalization, imputation, masks, and dimensionality reduction on training data only; compare metadata-only, spectra-only, and combined baselines; use group-aware validation for repeated galaxies; report per-Type metrics and class counts; and test sensitivity to the Early/Late boundary.

## 13. Export a reproducible metadata-to-spectrum manifest

In [ ]:
manifest=metadata[['galaxy_key',name_col,type_col,ttype_col,'broad_type']].copy()
manifest['archive_member']=manifest.galaxy_key.map(member_index); manifest['has_spectrum']=manifest.archive_member.notna()
manifest.to_csv(OUTPUT_DIR/'galaxy_spectrum_manifest.csv',index=False)
display(manifest.head())

## References

- [Astropy FITS documentation](https://docs.astropy.org/en/stable/io/fits/index.html)
- [Astropy FITS table tutorial](https://learn.astropy.org/tutorials/FITS-tables.html)
- [SDSS MaNGA DR17 overview](https://www.sdss4.org/dr17/manga/)

Review the FITS headers and project documentation for definitive units, masks, aperture definitions, and provenance.